# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlachiranjeevi/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Unit of analysis
One row represents one unique, pseudonymized content item for a client.

### Time window
The dataset contains trailing 90-day performance metrics, along with comparisons between the latest 30 days and the previous 30 days. The CSV does not contain a calendar reporting-date column, so an exact calendar start and end date cannot be established from this file. `days_since_last_update` represents elapsed days since the last update, not a reporting date.

### Verification result
The dataset contains 30,000 rows, 30,000 unique content IDs, zero duplicate content IDs, and 32 unique clients. Therefore, the verified row grain is one row per content item.

In [1]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Unique content IDs:', df['content_id'].nunique())
print('Duplicate content IDs:', df['content_id'].duplicated().sum())
print('Unique clients:', df['client_id'].nunique())

assert len(df) == 30000
assert df['content_id'].nunique() == len(df)
assert df['content_id'].duplicated().sum() == 0

print('\n90-day and 30-day metric fields:')
print([c for c in df.columns if any(x in c for x in ['90d', 'last_30d', 'prev_30d'])])

print('\nCalendar date columns:', [c for c in df.columns if 'date' in c.lower()])


Rows: 30000
Columns: 44
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32

90-day and 30-day metric fields:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

Calendar date columns: ['days_since_last_update']


## 2. Fields: feature / label / context / excluded

The fields are classified according to their role in a possible content-performance prediction task.

### Features ? candidate inputs
These describe content characteristics, search demand, competition, and content age. They may be considered as inputs, provided they are available before the prediction period.

- `search_volume`
- `competition`
- `competition_level`
- `cpc`
- `content_type`
- `main_intent`
- `word_count`
- `char_count`
- `content_age_days`
- `age_tier`
- `age_tier_order`
- `days_since_last_update`
- `freshness_tier`
- `word_count_tier`
- `char_count_tier`

### Label / proxy
These describe the trend outcome or a measure derived from it. They must not be used as input features.

- `trend_direction` ? categorical trend outcome
- `trend_pct` ? numeric trend measure

The dataset does not contain an explicit `is_declining_label` column.

### Context
These identify the content item and its client. They are useful for grouping, joining, and client-level train/test splitting, but must not be model features.

- `content_id`
- `client_id`

### Excluded
These fields are excluded from the initial candidate feature set because they describe observed performance outcomes, rates, or derived performance tiers. Using them may introduce target leakage if the prediction is intended to forecast future performance. `provider_used` and `model_used` are generation-system metadata, not content characteristics needed for this initial prediction.

- `provider_used`
- `model_used`
- `impressions_90d`
- `clicks_90d`
- `pageviews_90d`
- `sessions_90d`
- `users_90d`
- `engaged_sessions_90d`
- `ai_sessions_90d`
- `scroll_events_90d`
- `days_with_impressions`
- `days_with_sessions`
- `impressions_last_30d`
- `clicks_last_30d`
- `sessions_last_30d`
- `impressions_prev_30d`
- `clicks_prev_30d`
- `sessions_prev_30d`
- `ctr`
- `avg_position`
- `engagement_rate`
- `scroll_rate`
- `ai_traffic_pct`
- `impression_tier`
- `position_tier`

### Missing-value handling
Missing values must not automatically be replaced with zero. Missingness is concentrated in keyword, content-size, provider/model, and trend fields. Missingness should be examined by `content_type`; indicator fields such as `has_word_count` may be considered where appropriate. Also, `avg_position = 0` means no position data, not a real rank of zero.

### Leakage limitation
This is a conservative initial classification. Before training, the prediction target and prediction time must be defined. Candidate features must be available before that time. The observed performance fields above should not be used as predictors for a future-performance task.


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

field_groups = {
    "Feature": [
        "search_volume", "competition", "competition_level", "cpc",
        "content_type", "main_intent", "word_count", "char_count",
        "content_age_days", "age_tier", "age_tier_order",
        "days_since_last_update", "freshness_tier",
        "word_count_tier", "char_count_tier"
    ],
    "Label / proxy": [
        "trend_direction", "trend_pct"
    ],
    "Context": [
        "content_id", "client_id"
    ],
    "Excluded": [
        "provider_used", "model_used",
        "impressions_90d", "clicks_90d", "pageviews_90d",
        "sessions_90d", "users_90d", "engaged_sessions_90d",
        "ai_sessions_90d", "scroll_events_90d",
        "days_with_impressions", "days_with_sessions",
        "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
        "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
        "ctr", "avg_position", "engagement_rate", "scroll_rate",
        "ai_traffic_pct", "impression_tier", "position_tier"
    ]
}

# Verify that every dataset column is classified exactly once.
all_classified = [col for fields in field_groups.values() for col in fields]
duplicates = sorted({col for col in all_classified if all_classified.count(col) > 1})
missing_from_groups = sorted(set(df.columns) - set(all_classified))
unknown_fields = sorted(set(all_classified) - set(df.columns))

print("FIELD CLASSIFICATION COUNTS")
for group, fields in field_groups.items():
    print(f"{group}: {len(fields)}")

print("\nTotal classified:", len(all_classified))
print("Dataset columns:", len(df.columns))
print("Duplicate classifications:", duplicates)
print("Unclassified dataset columns:", missing_from_groups)
print("Unknown fields:", unknown_fields)

assert not duplicates, "A field appears in more than one group."
assert not missing_from_groups, "Some dataset fields are not classified."
assert not unknown_fields, "A classified field is not in the dataset."
assert len(all_classified) == len(df.columns)

print("\nMISSINGNESS BY CONTENT TYPE")
for col in ["search_volume", "competition", "cpc", "main_intent",
            "word_count", "char_count", "provider_used", "model_used",
            "trend_pct"]:
    print(f"\n{col}:")
    print(df.groupby("content_type", dropna=False)[col].agg(
        total="size", missing=lambda s: s.isna().sum()
    ).to_string())

print("\nAVG_POSITION ZERO COUNT:", int((df["avg_position"] == 0).sum()))
print("\nPASS: Every one of the 44 columns is classified exactly once.")


FIELD CLASSIFICATION COUNTS
Feature: 15
Label / proxy: 2
Context: 2
Excluded: 25

Total classified: 44
Dataset columns: 44
Duplicate classifications: []
Unclassified dataset columns: []
Unknown fields: []

MISSINGNESS BY CONTENT TYPE

search_volume:
                    total  missing
content_type                      
comparison article    697        0
feedly article       2096     2096
keyword article     27207      372

competition:
                    total  missing
content_type                      
comparison article    697        0
feedly article       2096     2096
keyword article     27207      372

cpc:
                    total  missing
content_type                      
comparison article    697        0
feedly article       2096     2096
keyword article     27207      372

main_intent:
                    total  missing
content_type                      
comparison article    697        0
feedly article       2096     2096
keyword article     27207      278

word_count:
   

## 3. Verify it with queries (grain, counts, missing values, windows)

### Grain and uniqueness
The dataset contains one row per unique content item. Verify the row count, unique content IDs, duplicate IDs, and number of clients.

### Counts by category
Check how many records belong to each content type and how many unique content items each client has.

### Missing values
Measure missing values across all columns and examine missingness by content type. Missing values are not automatically equivalent to zero.

### Metric windows
Inspect the available trailing 90-day metrics and the latest 30-day versus previous 30-day metrics. These are aggregate windows, not calendar-date columns.

### Special handling
`avg_position = 0` represents no position data and must be treated as missing for analysis. The CSV does not provide calendar reporting dates, so exact reporting start and end dates cannot be verified from this file.


In [3]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("1. GRAIN AND UNIQUENESS")
print("Rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", int(df["content_id"].duplicated().sum()))
print("Unique clients:", df["client_id"].nunique())

assert df["content_id"].nunique() == len(df)
assert df["content_id"].duplicated().sum() == 0

print("\n2. RECORD COUNTS BY CONTENT TYPE")
print(df["content_type"].value_counts(dropna=False).to_string())

print("\n3. CONTENT ITEMS PER CLIENT")
client_counts = df.groupby("client_id")["content_id"].nunique()
print("Clients:", len(client_counts))
print("Minimum content items per client:", int(client_counts.min()))
print("Maximum content items per client:", int(client_counts.max()))
print("Total content items across clients:", int(client_counts.sum()))

print("\n4. MISSING VALUES OVERALL")
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string() if len(missing) else "No missing values")

print("\n5. MISSING VALUES BY CONTENT TYPE")
for col in df.columns:
    grouped = df.groupby("content_type", dropna=False)[col].apply(
        lambda s: int(s.isna().sum())
    )
    if grouped.sum() > 0:
        print(f"\n{col}:")
        print(grouped.to_string())

print("\n6. PERFORMANCE METRIC WINDOWS")
window_cols = [
    "impressions_90d", "clicks_90d", "sessions_90d",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"
]
for col in window_cols:
    print(
        f"{col}: min={df[col].min()}, "
        f"max={df[col].max()}, "
        f"mean={df[col].mean():.2f}"
    )

print("\n7. AVG_POSITION SPECIAL CASE")
zero_positions = int((df["avg_position"] == 0).sum())
print("Rows where avg_position is zero:", zero_positions)
print("Rows where avg_position is non-zero:", int((df["avg_position"] != 0).sum()))

print("\n8. CALENDAR DATE CHECK")
date_columns = [c for c in df.columns if "date" in c.lower()]
print("Columns containing 'date':", date_columns)
print("days_since_last_update is a duration, not a calendar reporting date.")

assert len(df) == 30000
assert df["content_id"].nunique() == 30000
assert zero_positions == 1205

print("\nPASS: Section 3 verification checks completed.")


1. GRAIN AND UNIQUENESS
Rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
Unique clients: 32

2. RECORD COUNTS BY CONTENT TYPE
content_type
keyword article       27207
feedly article         2096
comparison article      697

3. CONTENT ITEMS PER CLIENT
Clients: 32
Minimum content items per client: 3
Maximum content items per client: 7008
Total content items across clients: 30000

4. MISSING VALUES OVERALL
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125

5. MISSING VALUES BY CONTENT TYPE

search_volume:
content_type
comparison article       0
feedly article        2096
keyword article        372

competition:
content_type
comparison article       0
feedly article        2096
k


word_count_tier:
content_type
comparison article       0
feedly article           0
keyword article       7699

char_count_tier:
content_type
comparison article       0
feedly article           0
keyword article       7699

scroll_rate:
content_type
comparison article      2
feedly article          1
keyword article       122

trend_pct:
content_type
comparison article       2
feedly article        1087
keyword article       2299

6. PERFORMANCE METRIC WINDOWS
impressions_90d: min=1, max=517715, mean=5200.37
clicks_90d: min=0, max=4178, mean=16.10
sessions_90d: min=1, max=4345, mean=37.07
impressions_last_30d: min=0, max=238796, mean=1429.06
clicks_last_30d: min=0, max=1176, mean=4.93
sessions_last_30d: min=0, max=1081, mean=14.11
impressions_prev_30d: min=0, max=218786, mean=1783.08
clicks_prev_30d: min=0, max=1627, mean=5.44
sessions_prev_30d: min=0, max=4247, mean=10.28

7. AVG_POSITION SPECIAL CASE
Rows where avg_position is zero: 1205
Rows where avg_position is non-zero: 28795

8


PASS: Section 3 verification checks completed.


## 4. Data limits

The starter CSV is useful for exploring content-level search and performance patterns, but it has limitations that affect modeling and interpretation.

### 1. Uneven client representation
There are 32 clients, but their content counts range from 3 to 7,008. A model trained on this data may be dominated by clients with many records. Use client-level grouping for train/test splitting rather than randomly splitting individual rows.

### 2. Missing values depend on content type
Some fields are systematically unavailable for particular content types. For example, search-volume, competition, and CPC fields are missing for all Feedly articles. Word-count and character-count fields are missing for many keyword articles. Missingness should not be blindly replaced with zero because that can create misleading signals.

### 3. No calendar reporting dates
The CSV contains aggregate 90-day metrics and latest-versus-previous 30-day metrics, but no calendar reporting-date column. `days_since_last_update` is a duration, not a reporting date. Exact reporting start and end dates cannot be established from this CSV.

### 4. Target and leakage limitations
The CSV contains `trend_direction` and `trend_pct`, but no separate `is_declining_label` column. The trend fields are derived and must not be used as input features when predicting that trend. The prediction target and prediction horizon still need to be explicitly defined before model training.

### 5. Performance metrics are observed outcomes
Impressions, clicks, sessions, engagement, CTR, position, and related tiers describe observed performance. They should not be used as predictors for a future-performance task unless their measurement window is strictly before the prediction point.

### 6. Position values require special handling
There are 1,205 rows where `avg_position` is zero. Zero represents no position data, not a genuine search ranking of zero. Treat these values as missing for position analysis.

### 7. Starter dataset scope
This file contains 30,000 content-item rows across 32 clients. It is a starter dataset, not the full warehouse panel. It does not provide the daily report-date history or query-level records described in the warehouse documentation.

### What this means for later modeling
Define the prediction target and prediction time first. Use only information available before that time, preserve client groups during splitting, investigate missingness by content type, and document the limitations of the starter CSV. Do not claim that this dataset establishes causal effects or exact calendar trends.


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

client_counts = df.groupby("client_id")["content_id"].nunique()
feedly = df[df["content_type"] == "feedly article"]
keyword = df[df["content_type"] == "keyword article"]

print("DATA LIMITS ? VERIFICATION")
print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Minimum content items per client:", int(client_counts.min()))
print("Maximum content items per client:", int(client_counts.max()))

print("\nMISSINGNESS EXAMPLES")
print("Feedly articles missing search_volume:",
      int(feedly["search_volume"].isna().sum()), "/", len(feedly))
print("Keyword articles missing word_count:",
      int(keyword["word_count"].isna().sum()), "/", len(keyword))

print("\nDATE AND TARGET LIMITS")
print("Calendar reporting date available:", any(
    c.lower() in ["date", "report_date", "reporting_date"] for c in df.columns
))
print("Explicit is_declining_label available:", "is_declining_label" in df.columns)
print("Trend fields available:",
      [c for c in ["trend_direction", "trend_pct"] if c in df.columns])

print("\nPOSITION LIMIT")
print("avg_position == 0:", int((df["avg_position"] == 0).sum()))

print("\nWINDOW FIELDS")
print([c for c in df.columns if "90d" in c or "last_30d" in c or "prev_30d" in c])

assert len(df) == 30000
assert df["client_id"].nunique() == 32
assert int((df["avg_position"] == 0).sum()) == 1205
assert "is_declining_label" not in df.columns

print("\nPASS: Section 4 data-limit checks completed.")


DATA LIMITS ? VERIFICATION
Rows: 30000
Clients: 32
Minimum content items per client: 3
Maximum content items per client: 7008

MISSINGNESS EXAMPLES
Feedly articles missing search_volume: 2096 / 2096
Keyword articles missing word_count: 7699 / 27207

DATE AND TARGET LIMITS
Calendar reporting date available: False
Explicit is_declining_label available: False
Trend fields available: ['trend_direction', 'trend_pct']

POSITION LIMIT
avg_position == 0: 1205

WINDOW FIELDS
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']

PASS: Section 4 data-limit checks completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.